In [1]:
# importing libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns


train_transaction = pd.read_csv("../data/train_transaction.txt")
train_identity=pd.read_csv("../data/train_identity.txt")
train_df =  train_transaction.merge(train_identity , on="TransactionID" , how="left")

test_identity = pd.read_csv("../data/test_identity.csv")
test_identity.columns = test_identity.columns.str.replace("-" , "_")
test_transaction = pd.read_csv("../data/test_transaction.txt")
test_df = test_transaction.merge(test_identity , on="TransactionID" , how="left")


In [2]:
train_seconds = train_df["TransactionDT"]
train_df["Transaction_Day"] = (train_seconds // 86400).astype(int)
train_df["Transaction_Hour"] = ((train_seconds % 86400) // 3600).astype(int)
train_df["Transaction_Minute"] = ((train_seconds % 3600) // 60).astype(int)
train_df["Transaction_week"] = (train_df["Transaction_Day"] // 7).astype(int)
train_df["Transaction_Time_Of_Day"] = (train_seconds % 86400)


test_seconds = test_df["TransactionDT"]
test_df["Transaction_Day"] = (test_seconds // 86400).astype(int)
test_df["Transaction_Hour"] = ((test_seconds % 86400) // 3600).astype(int)
test_df["Transaction_Minute"] = ((test_seconds % 3600) // 60).astype(int)
test_df["Transaction_week"] = (test_df["Transaction_Day"] // 7).astype(int)
test_df["Transaction_Time_Of_Day"] = (test_seconds % 86400)

In [3]:
train_df["Hour_sin"] = np.sin(2 * np.pi * train_df["Transaction_Hour"] / 24)
train_df["Hour_cos"] = np.cos(2 * np.pi * train_df["Transaction_Hour"] / 24)

test_df["Hour_sin"] = np.sin(2 * np.pi * test_df["Transaction_Hour"] / 24)
test_df["Hour_cos"] = np.cos(2 * np.pi * test_df["Transaction_Hour"] / 24)

In [4]:
train_df["log_TransactionAmt"] = np.log1p(train_df["TransactionAmt"])

test_df["log_TransactionAmt"] = np.log1p(test_df["TransactionAmt"])

In [5]:
train_df["AmountBucket"] = pd.cut(
    train_df["TransactionAmt"] , 
    bins=[0 , 10 , 50 , 100 , 500 , 1000 , float("inf")] , 
    labels=False
)

test_df["AmountBucket"] = pd.cut(
    test_df["TransactionAmt"] , 
    bins=[0 , 10 , 50 , 100 , 500 , 1000 , float("inf")] , 
    labels=False
)

In [6]:
train_df["Missing_Count"] = train_df.isna().sum(axis=1)
test_df["Missing_Count"] = test_df.isna().sum(axis=1)

In [7]:
train_df["Address_Missing"] = (train_df["addr1"].isna() | train_df["addr2"].isna()).astype(int)
test_df["Address_Missing"] = (test_df["addr1"].isna() | test_df["addr2"].isna()).astype(int)

In [8]:
train_df["email_match"] = (train_df["P_emaildomain"] == train_df["R_emaildomain"]).astype(int)
test_df["email_match"] = (test_df["P_emaildomain"] == test_df["R_emaildomain"]).astype(int)

In [9]:
card_stats = (train_df.groupby('card1')['TransactionAmt'].agg(['mean' , 'std']))

train_df = train_df.copy()
train_df["TransactionAmt_card1_z"] = ((train_df['TransactionAmt'] - train_df['card1'].map(card_stats['mean'])) / (train_df['card1'].map(card_stats['std']) + 1e-5))


card_stats_test = (test_df.groupby('card1')['TransactionAmt'].agg(['mean' , 'std']))

test_df = test_df.copy()
test_df["TransactionAmt_card1_z"] = ((test_df['TransactionAmt'] - test_df['card1'].map(card_stats_test['mean'])) / (test_df['card1'].map(card_stats_test['std']) + 1e-5))

In [10]:
def feature_cents(df):
    return (np.modf(df['TransactionAmt'])[0] * 1000).astype(np.uint16)
train_df['Cents'] = feature_cents(train_df)
test_df['Cents'] = feature_cents(test_df)

In [11]:
def feature_ProductCD_W_cents(df):
    a = ((df['Cents'] == 0) | (df['Cents'] == 500) | (df['Cents'] == 950))
    b = df['ProductCD'] == 'W'
    return (a & b).astype(np.uint8)

train_df['Cents_ProductCD_W'] = feature_ProductCD_W_cents(train_df)
test_df['Cents_ProductCD_W'] = feature_ProductCD_W_cents(test_df)

In [12]:
COL_ID = 'TransactionID'
COL_DT = 'TransactionDT'
COL_AMOUNT = 'TransactionAmt'
COL_TARGET = 'isFraud'

interactions = [
    'card1__addr1', 'card1__card5', 'card2__dist1', 'card2__id_20', 'card5__P_emaildomain',
    'card1__card2__card3__card5', 'card1__card2__card3__card5__addr1__addr2'
]
rename = {
    'card1__card2__card3__card5': 'uuid2',
    'card1__card2__card3__card5__addr1__addr2': 'uuid3'
}

encode_freq_cols = ['card1__addr1', 'card1__card5', 'uuid3']
groups = [
    (['uuid2', 'ProductCD'], COL_AMOUNT, 'mean'),
    (['uuid2', 'addr1'], 'dist2', 'mean'),
]
drop = ['uuid2', 'uuid3']

for inter in interactions:
    cols = inter.split('__')
    col_name = rename.get(inter , inter)

    for df in [train_df , test_df]:
        df[col_name] = df[cols].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)  


In [13]:
groups = [
    (['uuid2', 'ProductCD'], COL_AMOUNT, 'mean'),
    (['uuid2', 'addr1'], 'dist2', 'mean'),
]
for df in [train_df , test_df]:
    for group, target, aggs in groups:
        col = f'{target}_{aggs}_by_{"_".join(group)}'
        df[col] = df.groupby(group)[target].transform(aggs)       

In [14]:
na_cols = ['D2', 'D3', 'D5', 'D6', 'D7', 'D13', 'D14', 'M4', 'M5', 'id_16']

for df in [train_df , test_df]:
    for col in na_cols:
        df[col+'_na'] = df[col].isna().astype(np.uint8)

In [15]:
EMAIL_MAP = {
    'gmail': 'google', 'att.net': 'att', 'twc.com': 'spectrum', 'scranton.edu': 'other', 'optonline.net': 'other', 
    'hotmail.co.uk': 'microsoft', 'comcast.net': 'other', 'yahoo.com.mx': 'yahoo', 'yahoo.fr': 'yahoo', 
    'yahoo.es': 'yahoo', 'charter.net': 'spectrum', 'live.com': 'microsoft', 'aim.com': 'aol', 
    'hotmail.de': 'microsoft', 'centurylink.net': 'centurylink', 'gmail.com': 'google', 'me.com': 'apple', 
    'earthlink.net': 'other', 'gmx.de': 'other', 'web.de': 'other', 'cfl.rr.com': 'other', 'hotmail.com': 'microsoft', 
    'protonmail.com': 'other', 'hotmail.fr': 'microsoft', 'windstream.net': 'other', 'outlook.es': 'microsoft', 
    'yahoo.co.jp': 'yahoo', 'yahoo.de': 'yahoo', 'servicios-ta.com': 'other', 'netzero.net': 'other', 
    'suddenlink.net': 'other', 'roadrunner.com': 'other', 'sc.rr.com': 'other', 'live.fr': 'microsoft', 
    'verizon.net': 'yahoo', 'msn.com': 'microsoft', 'q.com': 'centurylink', 'prodigy.net.mx': 'att', 
    'frontier.com': 'yahoo', 'anonymous.com': 'other', 'rocketmail.com': 'yahoo', 'sbcglobal.net': 'att', 
    'frontiernet.net': 'yahoo', 'ymail.com': 'yahoo', 'outlook.com': 'microsoft', 'mail.com': 'other', 
    'bellsouth.net': 'other', 'embarqmail.com': 'centurylink', 'cableone.net': 'other', 'hotmail.es': 'microsoft', 
    'mac.com': 'apple', 'yahoo.co.uk': 'yahoo', 'netzero.com': 'other', 'yahoo.com': 'yahoo', 'live.com.mx': 'microsoft', 
    'ptd.net': 'other', 'cox.net': 'other', 'aol.com': 'aol', 'juno.com': 'other', 'icloud.com': 'apple'
}

In [16]:
def parse_email_suffix(email):
    if email is np.nan:
        return 0, np.nan, np.nan
    parts = email.split('.') 
    middle = parts[-2] if len(parts) > 2 else parts[-1]
    last = parts[-1]
    return len(parts), middle, last

In [17]:
for df in [train_df , test_df]:
    for col in ['P_emaildomain', 'R_emaildomain']:
        temp = df[col].apply(lambda x: parse_email_suffix(x))
        df[col+'_parts'] = temp.map(lambda x: x[0])
        df[col+'_sfx2']  = temp.map(lambda x: x[1])
        df[col+'_sfx1']  = temp.map(lambda x: x[2])
        df[col+'_bin'] = df[col].map(EMAIL_MAP)    

In [18]:
from functools import partial
def str_contains(df, col, val):
    return df[col].str.contains(val, case=False).fillna(False)


for df in [train_df ,test_df ]:
    col = 'Browser'
    df[col] = -1
    search = partial(str_contains, df, 'id_31')
    for i, browser in enumerate(['chrome', 'safari', 'ie ', 'firefox', 'edge', 'samsung', 'opera']):
        df.loc[search(browser), [col]] = i
    df.loc[(df['id_31'] == 'ie'), [col]] = 2
    
for df in [train_df , test_df]:
    col = 'OS'
    df[col] = -1
    search = partial(str_contains, df, 'id_30')
    for i, os_name in enumerate(['windows', 'ios', 'mac', 'android', 'linux']):
        df.loc[search(os_name), [col]] = i

C:\Users\user\AppData\Local\Temp\ipykernel_9164\716319507.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[col].str.contains(val, case=False).fillna(False)
C:\Users\user\AppData\Local\Temp\ipykernel_9164\716319507.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df[col].str.contains(val, case=False).fillna(False)
C:\Users\user\AppData\Local\Temp\ipykernel_9164\716319507.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=Fa

In [19]:
for col in ['M1','M2','M3','M5','M6','M7','M8','M9']:
    for df in [train_df , test_df]:
        df[col] = df[col].map({ 'T': 1, 'F': 0 })

In [20]:
train_df.to_pickle("../data/train.pkl")
test_df.to_pickle("../data/test.pkl")